## Implement PKCS#7 padding

In [ ]:
pt = "YELLOW SUBMARINE"
offset = abs(len(pt) - 20)
ct = pt.encode() + (offset.to_bytes() * offset)
print(ct)

## Implement CBC mode

In [ ]:
import base64
from Crypto.Cipher import AES

In [ ]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/10.txt") as f:
    b64_data = "".join(
        line.strip()
        for line in f
    )

ct = base64.b64decode(b64_data)
key = b"YELLOW SUBMARINE"
iv  = (len(key)*0).to_bytes() * len(key)
blocks = [ct[i:i+16] for i in range(0,len(ct),16)]

pt = b""
cipher = AES.new(key, AES.MODE_ECB)
for i in range(len(blocks)):
    ci = cipher.decrypt(blocks[i])
    if i == 0:
        pt += fixed_xor(ci, iv)
        continue
    pt += fixed_xor(ci, blocks[i-1])

print(pt.decode())    

## An ECB/CBC detection oracle


In [ ]:
import os
import random
from Crypto.Cipher import AES

In [ ]:
def salting(pt: bytes):
    count = random.randrange(5,11)
    prefix = os.urandom(count)
    suffix = os.urandom(count)
    return prefix + pt + suffix

def pkcs7(pt:bytes):
    offset = 16 - (len(pt)%16)
    ct = pt + (offset.to_bytes() * offset)
    return ct

def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

def aes_ecb(pt: bytes, key: bytes):
    plain = b""
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def aes_cbc(pt: bytes, key: bytes):
    iv = os.urandom(16)
    plain = salting(pt)
    if len(plain) % 16 != 0:
        plain = pkcs7(plain)
    
    blocks = [plain[i:i+16] for i in range(0,len(plain),16)]
    cipher = AES.new(key, AES.MODE_ECB)
    for i in range(len(blocks)):
        if i == 0:
            ci = fixed_xor(blocks[i], iv)
            blocks[i] = cipher.encrypt(ci)
            continue
        ci = fixed_xor(blocks[i-1], blocks[i])
        blocks[i] = cipher.encrypt(ci)
    
    ct = b"".join(blocks)
    return ct

In [ ]:
def aes_oracle(pt: str):
    enc_pt = pt.encode()
    
    key = os.urandom(16)
    aes_mode = random.randrange(1,3)
    ct = b""
    match aes_mode:
        case 1:
            # print("ECB")
            ct = aes_ecb(enc_pt, key)
        case 2:
            # print("CBC")
            ct = aes_cbc(enc_pt, key)
    return ct

In [ ]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

In [ ]:
pt = "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"
ct = aes_oracle(pt)
print(ct)

blocks = split_block(ct)
repeated = check_repeated(blocks)
if repeated > 0:
    print(f"Its ECB")
else:
    print("Its CBC")

## Byte-at-a-time ECB decryption (Simple)
 

In [ ]:
import os
import random
import base64
from Crypto.Cipher import AES

KEY = os.urandom(16)

In [ ]:
def salting(pt: bytes):
    salt = "Um9sbGluJyBpbiBteSA1LjAKV2l0aCBteSByYWctdG9wIGRvd24gc28gbXkgaGFpciBjYW4gYmxvdwpUaGUgZ2lybGllcyBvbiBzdGFuZGJ5IHdhdmluZyBqdXN0IHRvIHNheSBoaQpEaWQgeW91IHN0b3A/IE5vLCBJIGp1c3QgZHJvdmUgYnkK"
    salt_enc = base64.b64decode(salt)
    return pt + salt_enc

def pkcs7(pt:bytes):
    offset = 16 - (len(pt)%16)
    ct = pt + (offset.to_bytes() * offset)
    return ct

def aes_ecb(pt: bytes, key: bytes):
    plain = b""
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def aes_oracle(pt: bytes):
    enc_pt = pt
    salting_pt = salting(enc_pt)
    key = KEY
    ct = ct = aes_ecb(salting_pt, key)
    return ct

In [ ]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

def find_block_size():
    base_len = 0
    for i in range(0,64):
        pt = b'A'*i
        ct = aes_oracle(pt)
        if i == 0:
            base_len += len(ct)
            continue
        if len(ct) > base_len:
            return len(ct) - base_len
    raise Exception("Block size not found")

In [ ]:
block_size = find_block_size()
print(f"Block size: {block_size}")

In [ ]:
pt = b'A'*(block_size*3)
ct = aes_oracle(pt)
blocks = split_block(ct, block_size)
repeated = check_repeated(blocks)

if repeated > 0:
    print("its ECB")

In [ ]:
sentence_cipher = {}
for w in range(256):
    pt = b'A'*15 + bytes([w])
    ct = aes_oracle(pt)
    sentence_cipher[ct[:16]] = bytes([w])

# for k in sentence_cipher:
#     print(k, sentence_cipher[k])

In [ ]:
short_ct = aes_oracle(b'A'*15)[:16]
first_byte = sentence_cipher[short_ct]
print(f"Byte secret pertama: {first_byte}")

In [ ]:
known_pt = b""
while True:
    pad_len = (block_size - 1) - (len(known_pt) % block_size)
    prefix = b"A" * pad_len
    target_block_idx = len(known_pt) // block_size

    sentence_cipher = {}
    for w in range(256):
        cand = prefix + known_pt + bytes([w])
        ct = aes_oracle(cand)
        sentence_cipher[ct[:16]] = bytes([w])
        dict_key = split_block(ct, block_size)[target_block_idx]
        sentence_cipher[dict_key] = bytes([w])

    ct_short = aes_oracle(prefix)
    target_ct_block = split_block(ct_short, block_size)[target_block_idx]

    if target_ct_block in sentence_cipher:
        known_pt += sentence_cipher[target_ct_block]
    else:
        break 
print(known_pt.decode())

## ECB cut-and-paste


In [ ]:
routine = "foo=bar&baz=qux&zap=zazzle"
raw = (routine.replace("&","=")).split("=")
dictionary = {}
for i in range(len(raw)):
    if i % 2 == 0:
        # print(raw[i])
        dictionary[raw[i]] = raw[i+1]

print(dictionary)

In [ ]:
KEY = os.urandom(16)

In [ ]:
def profile_for(profile: str):
    email = profile.replace("&","").replace("=","")
    uid = 10
    role = 'user'

    return f"email={email}&uid={uid}&role={role}"

def pkcs7(pt:bytes, block_size: int = 16):
    offset = block_size - (len(pt)%block_size)
    return pt + (offset.to_bytes() * offset)

def encrypt_profile(pt: bytes):
    plain = b""
    key = KEY
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def decrypt_profile(profile_enc: bytes):
    cipher = AES.new(KEY, AES.MODE_ECB)
    dec = cipher.decrypt(profile_enc)
    pad_len = dec[-1]
    return dec[:-pad_len]

def parse_cookies(dec: bytes):
    routine = dec.decode()
    raw = (routine.replace("&","=")).split("=")
    dictionary = {}
    for i in range(len(raw)):
        if i % 2 == 0:
            # print(raw[i])
            dictionary[raw[i]] = raw[i+1]
    return dictionary

profile = profile_for("foo@bar.com")
encrypt = encrypt_profile(profile.encode())
decrypt = decrypt_profile(encrypt)
cookies = parse_cookies(decrypt)
print(encrypt)
print(decrypt)
print(cookies)

email=      -> 6
email       -> x
&uid=       -> 5
uid         -> 2
&role=      -> 6
total = 19

x + 19 ≡ 0 (mod 16)
x = 13
normal ct
block1 = email=AAAAA@bar.com
block2 = &uid=10&role=user+padd

spoof ct
block1 = email=admin

In [ ]:
email = "AAAAA@bar.com"
profile = profile_for(email)
encrypt = encrypt_profile(profile.encode())
print(profile[:32])
print(encrypt)
print()

payload = 'B'*10+"admin" + '\x0b'*11 
profile_admin = profile_for(payload)
encrypt_admin = encrypt_profile(profile_admin.encode())
crafted = encrypt[:32] + encrypt_admin[16:32]
decrypt_admin = decrypt_profile(crafted)
cookies_admin = parse_cookies(decrypt_admin)
print(cookies_admin)

## Byte-at-a-time ECB decryption (Harder)

In [ ]:
import os
import random
import base64
from Crypto.Cipher import AES

In [ ]:
KEY = os.urandom(16)
PREFIX = os.urandom(random.randint(16,128))

In [ ]:
def _prefix(pt: bytes):
    ct = PREFIX + pt
    return ct

def _suffix(pt: bytes):
    salt = "Um9sbGluJyBpbiBteSA1LjAKV2l0aCBteSByYWctdG9wIGRvd24gc28gbXkgaGFpciBjYW4gYmxvdwpUaGUgZ2lybGllcyBvbiBzdGFuZGJ5IHdhdmluZyBqdXN0IHRvIHNheSBoaQpEaWQgeW91IHN0b3A/IE5vLCBJIGp1c3QgZHJvdmUgYnkK"
    salt_enc = base64.b64decode(salt)
    return pt + salt_enc

def pkcs7(pt:bytes):
    offset = 16 - (len(pt)%16)
    ct = pt + (offset.to_bytes() * offset)
    return ct

def aes_ecb(pt: bytes, key: bytes):
    plain = b""
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def ecb_oracle(pt: bytes):
    prefix_enc = _prefix(pt)
    suffix_enc = _suffix(prefix_enc)
    key = KEY
    ct = aes_ecb(suffix_enc, key)
    return ct

In [ ]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

In [ ]:
for i in range(40,43):
    input_user = b'A'*(i)
    ct = ecb_oracle(input_user)
    blocks = split_block(ct)
    repeated = check_repeated(blocks)

    if repeated > 0:
        print("its ecb")
        print(i)
        for block in blocks:
            print(block)
    else:
        print(i)
        print("increase your payloads size...")
    print()

r = secret prefix
A = (16 - r) + 32 = 48 - r

n  = 40 tidak muncul block berulang
40 >= 48 - r
-8 >= -r
r  >= 8

n = 41 muncul block berulang pertama
41 >= 48 - r
-7 >= -r
r  >= 7

n = 42 muncul block berulang lagi
42 >= 48 - r
-6 >= -r
r  >= 6

r = 7
block 1 = 16 byte
block 2 = 16 byte
block 3 = 16 byte
block 4 = 16 byte
block 5 = 16 byte - 7 = 9
block 6 = 16 byte [payload]

secret block = (16.4) + 9

In [ ]:
payload = b'A'*9
block_size = 16

In [ ]:
known_pt = b""
while True:
    pad_len = (block_size - 1) - (len(known_pt) % block_size)
    pad_align = b"A" * pad_len
    target_block_idx = 5 + len(known_pt) // block_size

    sentence_cipher = {}
    for w in range(256):
        cand = pad_align + known_pt + bytes([w])
        ct = ecb_oracle(payload + cand)
        dict_key = split_block(ct, block_size)[target_block_idx]
        sentence_cipher[dict_key] = bytes([w])

    ct_short = ecb_oracle(payload + pad_align)
    target_ct_block = split_block(ct_short, block_size)[target_block_idx]

    if target_ct_block in sentence_cipher:
        known_pt += sentence_cipher[target_ct_block]
    else:
        break 
print(known_pt.decode())

## PKCS#7 padding validation
Write a function that takes a plaintext, determines if it has valid PKCS#7 padding, and strips the padding off.

The string:

"ICE ICE BABY\x04\x04\x04\x04"

... has valid padding, and produces the result "ICE ICE BABY".

The string:

"ICE ICE BABY\x05\x05\x05\x05"

... does not have valid padding, nor does:

"ICE ICE BABY\x01\x02\x03\x04"

If you are writing in a language with exceptions, like Python or Ruby, make your function throw an exception on bad padding.

Crypto nerds know where we're going with this. Bear with us.

In [ ]:
def check_pkcs7(pt: str):
    if not pt:
        raise ValueError("your data is empty lol...")
    b_pt = pt.encode()
    pad_len = b_pt[-1]

    if pad_len == 0 or pad_len > len(b_pt):
        raise ValueError("invalid padding length...")

    if b_pt[-pad_len:] != bytes([pad_len]*pad_len):
        raise ValueError("padding length inconsistent...")
    
    return b_pt[:-pad_len]

pt = "ICE ICE BABY\x04\x04\x04\x04"
check = check_pkcs7(pt)
print(check)

pt2 = "ICE ICE BABY\x01\x02\x03\x04"
check2 = check_pkcs7(pt2)
print(check2)

## CBC bitflipping attacks
Generate a random AES key.

Combine your padding code and CBC code to write two functions.

The first function should take an arbitrary input string, prepend the string:

"comment1=cooking%20MCs;userdata="

.. and append the string:

";comment2=%20like%20a%20pound%20of%20bacon"

The function should quote out the ";" and "=" characters.

The function should then pad out the input to the 16-byte AES block length and encrypt it under the random AES key.

The second function should decrypt the string and look for the characters ";admin=true;" (or, equivalently, decrypt, split the string on ";", convert each resulting string into 2-tuples, and look for the "admin" tuple).

Return true or false based on whether the string exists.

If you've written the first function properly, it should not be possible to provide user input to it that will generate the string the second function is looking for. We'll have to break the crypto to do that.

Instead, modify the ciphertext (without knowledge of the AES key) to accomplish this.

You're relying on the fact that in CBC mode, a 1-bit error in a ciphertext block:
- Completely scrambles the block the error occurs in
- Produces the identical 1-bit error(/edit) in the next ciphertext block.

### Stop and think for a second.
Before you implement this attack, answer this question: why does CBC mode have this property?

In [ ]:
import os
from Crypto.Cipher import AES

In [ ]:
KEY = os.urandom(16)

In [ ]:
def pkcs7(pt:bytes):
    offset = 16 - (len(pt)%16)
    ct = pt + (offset.to_bytes() * offset)
    return ct

def check_pkcs7(pt: bytes):
    if not pt:
        raise ValueError("your data is empty lol...")
    b_pt = pt
    pad_len = b_pt[-1]

    if pad_len == 0 or pad_len > len(b_pt):
        raise ValueError("invalid padding length...")

    if b_pt[-pad_len:] != bytes([pad_len]*pad_len):
        raise ValueError("padding length inconsistent...")
    
    return b_pt[:-pad_len]

def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

def cbc_encrypt(pt: bytes, key: bytes):
    iv = os.urandom(16)
    pt = pkcs7(pt)
    
    blocks = [pt[i:i+16] for i in range(0,len(pt),16)]
    cipher = AES.new(key, AES.MODE_ECB)
    for i in range(len(blocks)):
        if i == 0:
            ci = fixed_xor(blocks[i], iv)
            blocks[i] = cipher.encrypt(ci)
            continue
        ci = fixed_xor(blocks[i-1], blocks[i])
        blocks[i] = cipher.encrypt(ci)
    
    ct = b"".join(blocks)
    return iv + ct

def cbc_decrypt(ct: bytes, key: bytes):
    iv = ct[:16]
    ciphertext = ct[16:] 

    blocks = [ciphertext[i:i+16] for i in range(0,len(ciphertext),16)]
    cipher = AES.new(key, AES.MODE_ECB)
    tmp_block = b""
    for i in range(len(blocks)):
        tmp = blocks[i]
        if i == 0:
            tmp_block = tmp
            ci = cipher.decrypt(blocks[i])
            blocks[i] = fixed_xor(ci, iv)
            continue
        ci = cipher.decrypt(blocks[i])
        blocks[i] = fixed_xor(ci, tmp_block)
        tmp_block = tmp
    
    pt = b"".join(blocks)
    return pt

def encrypt_userdata(user_input: str):
    sanitize = user_input.replace(";", "%3B").replace("=","%3D")
    prefix = "comment1=cooking%20MCs;userdata="
    suffix = ";comment2=%20like%20a%20pound%20of%20bacon"
    pt = prefix.encode() + sanitize.encode() + suffix.encode() 
    ct = cbc_encrypt(pt, KEY)
    return ct
    
def decrypt_userdata(ct: bytes):
    pt = cbc_decrypt(ct, KEY)
    pt = check_pkcs7(pt)
    return pt

test1 = "hello;admin=true"
ct1 = encrypt_userdata(test1)
pt1 = decrypt_userdata(ct1)
print(ct1)
print(pt1)

block1 = comment1=cooking
block2 = %20MCs;userdata=
block3 = AAAAAAAAAAAAAAAA
block4 = :admin<true:comm
block5 = ent2=%20like%20a
block6 = %20pound%20of%20
block7 = bacon' + padd

so the target is block4

In [36]:
print(hex(0x3A ^ 0x1))
print(hex(0x3B ^ 0x1))

print(chr(0x3A ^ 0x1))
print(chr(0x3B ^ 0x1))

print(hex(0x3D ^ 0x1))
print(hex(0x3C ^ 0x1))

print(chr(0x3D ^ 0x1))
print(chr(0x3C ^ 0x1))

0x3b
0x3a
;
:
0x3c
0x3d
<
=


In [ ]:
raw_input = "AAAAAAAAAAAAAAAA:admin<true:"
raw_ct = encrypt_userdata(raw_input)

iv = raw_ct[:16]
blocks = [raw_ct[i:i+16] for i in range(16,len(raw_ct),16)]

debug = blocks[2]
print(blocks)

target = bytearray(blocks[2])
target[0] ^= 0x1
target[6] ^= 0x1
target[11] ^= 0x1
blocks[2] = bytes(target)

ct_tampered = iv + b"".join(blocks)
pt = decrypt_userdata(ct_tampered)
print(pt)

[b'\xe0\xc0\xc1\xce\xfd\xeb\xfcH\xa7\xc0\xd1M\xcf\xc7\x06U', b'h\xe2\t\xfe\x18-Yw\xd4&\xe6J\xf9\xe8\xcf~', b'\x82\xd0E\x1a\x089\x94\xe8\xd7#s[\xf3]\x8b!', b'\x8c\xda:\x98\xb9\xd0f\x94\x88N\xdf\xb9\x8b\xc3\xa8\x01', b'n\x07\x16\x0e\x8ft\x86u\x1d\xd9\xa0\x9d\x0f\xe0\x9a\xba', b'\xd2\x1b\xe2@\xea\xe9v\xa5\xcc\x92\xe5qf2k/', b'\xf8y\x1d\xc2\xaa\x17$\x0e\x18\xe3T\xa5\xe2\x056\x83']
b"comment1=cooking%20MCs;userdata=\xf1Bkn\x8e\xbc'\r\xcb\xb0\xaa\xfc\x00\xe2\xc7\xbb;admin=true;;comment2=%20like%20a%20pound%20of%20bacon"
